# Mini Projeto — Análise de Vendas de Halloween

Este projeto tem como objetivo analisar uma base fictícia de vendas de produtos relacionados ao Halloween.

Ao longo do projeto serão realizadas etapas de leitura dos dados, limpeza, validação, tratamento de datas e análise de vendas, buscando identificar padrões que possam apoiar decisões de negócio.

## Sprint 1 — Configuração e leitura dos dados

In [10]:
# Importar a biblioteca Pandas para trabalhar com os dados.

import pandas as pd

In [11]:
# Carregar a base de vendas preservando os dados inicialmente como texto.

df = pd.read_csv(
    "halloween_vendas.csv",
    sep=",",
    encoding="utf-8",
    dtype=str
)

In [12]:
# Conferir a quantidade de registros e visualizar as primeiras linhas.

print("Total de registros:", len(df))

display(df.head(3))

Total de registros: 1500


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


In [13]:
# Verificar os nomes das colunas disponíveis na base.

print(df.columns.tolist())

['pedido_id', 'data_venda', 'produto', 'categoria', 'canal_venda', 'cidade', 'quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct', 'status_pedido', 'publico', 'tamanho', 'data_entrega']


In [14]:
# Conferir a quantidade de valores ausentes em cada coluna.

print(df.isna().sum())

pedido_id           0
data_venda          0
produto             0
categoria          18
canal_venda         0
cidade              0
quantidade          0
preco_unitario     12
custo_unitario      7
desconto_pct        0
status_pedido       0
publico             0
tamanho             0
data_entrega      143
dtype: int64


In [15]:
# Verificar como os valores da coluna categoria aparecem na base.

print(df["categoria"].value_counts(dropna=False))

categoria
Fantasia            892
Acessorio           459
 Fantasia            48
  FANTASIA!!!        39
 Acessorio           28
NaN                  18
  ACESSORIO!!!       16
Name: count, dtype: int64


In [16]:
# Verificar como os valores da coluna canal_venda aparecem na base.

print(df["canal_venda"].value_counts(dropna=False))

canal_venda
Web              646
Loja física      535
Mercado          274
 web              20
 loja física      16
 mercado           9
Name: count, dtype: int64


## Sprint 2 — Limpeza e padronização dos textos

In [17]:
# Criar uma função para padronizar os valores da coluna categoria.

import re

def limpar_categoria(valor):
    if pd.isna(valor):
        return "sem categoria"

    valor = valor.strip()
    valor = valor.lower()
    valor = re.sub(r"[^a-zà-ÿ\s]", "", valor)

    return valor

In [18]:
# Aplicar a limpeza da categoria em uma nova coluna.

df["categoria_limpa"] = df["categoria"].apply(limpar_categoria)

In [19]:
# Conferir o resultado da padronização da categoria.

print(df["categoria_limpa"].value_counts(dropna=False))

categoria_limpa
fantasia         979
acessorio        503
sem categoria     18
Name: count, dtype: int64


In [20]:
# Criar uma função para padronizar os valores da coluna canal_venda.

def limpar_canal_venda(valor):
    valor = valor.strip()
    valor = valor.lower()

    if valor == "web":
        return "Web"
    elif valor == "loja física":
        return "Loja física"
    elif valor == "mercado":
        return "Mercado"

    return valor

In [ ]:
# Aplicar a limpeza do canal de venda em uma nova coluna e conferir o resultado da padronização.

df["canal_venda_limpo"] = df["canal_venda"].apply(limpar_canal_venda)
print(df["canal_venda_limpo"].value_counts(dropna=False))

canal_venda_limpo
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64


In [ ]:
# Contar quantos registros da categoria estavam diferentes do padrão esperado e 
# também quantos registros do canal de venda foram ajustados.

categorias_validas = ["Fantasia", "Acessorio"]

problema_categoria = ~df["categoria"].isin(categorias_validas)

quantidade_inconsistencias_categoria = problema_categoria.sum()

print(
    "Registros com inconsistências na categoria:",
    quantidade_inconsistencias_categoria
)


quantidade_corrigida_canal = (
    df["canal_venda"] != df["canal_venda_limpo"]
).sum()

print(
    "Registros ajustados em canal_venda:",
    quantidade_corrigida_canal
)

Registros com inconsistências na categoria: 149
Registros ajustados em canal_venda: 45


## Sprint 3 — Validação dos dados numéricos

In [28]:
# Converter quantidade e preço para formato numérico em novas colunas.

df["quantidade_num"] = pd.to_numeric(
    df["quantidade"],
    errors="coerce"
)

df["preco_unitario_num"] = pd.to_numeric(
    df["preco_unitario"],
    errors="coerce"
)

In [29]:
# Identificar registros com quantidade ou preço inválidos.

df["problema_quantidade"] = (
    df["quantidade_num"].isna()
    | (df["quantidade_num"] <= 0)
    | (df["quantidade_num"] % 1 != 0)
)

df["problema_preco"] = (
    df["preco_unitario_num"].isna()
    | (df["preco_unitario_num"] < 0)
)

In [30]:
# Contar quantos registros apresentam problemas numéricos.

print(
    "Problemas em quantidade:",
    df["problema_quantidade"].sum()
)

print(
    "Problemas em preço:",
    df["problema_preco"].sum()
)

Problemas em quantidade: 7
Problemas em preço: 12


In [31]:
# Identificar os pedidos que possuem algum problema de quantidade ou preço.

pedidos_com_problema = df[
    df["problema_quantidade"] | df["problema_preco"]
]

quantidade_pedidos_afetados = pedidos_com_problema["pedido_id"].nunique()

print(
    "Pedidos únicos afetados por problemas numéricos:",
    quantidade_pedidos_afetados
)

Pedidos únicos afetados por problemas numéricos: 19


In [32]:
# Visualizar os registros com problemas numéricos.

display(
    pedidos_com_problema[
        [
            "pedido_id",
            "quantidade",
            "preco_unitario",
            "problema_quantidade",
            "problema_preco"
        ]
    ]
)

,pedido_id,quantidade,preco_unitario,problema_quantidade,problema_preco
23,1500,3,NaN,False,True
164,630,-1,22,True,False
199,625,2,NaN,False,True
216,1000,1,NaN,False,True
239,1250,1,NaN,False,True
293,500,1,NaN,False,True
503,375,3,NaN,False,True
571,250,4,NaN,False,True
583,125,1,NaN,False,True
672,840,-1,23.32,True,False


## Sprint 4 — Regras de negócio e tratamento de datas

In [33]:
# Separar os pedidos sem data de entrega para testar a hipótese.

sem_entrega = df[df["data_entrega"].isna()]

sem_entrega_cancelados = sem_entrega[
    sem_entrega["status_pedido"] == "Cancelado"
]

sem_entrega_outros_status = sem_entrega[
    sem_entrega["status_pedido"] != "Cancelado"
]

In [34]:
# Comparar quantos pedidos sem entrega estão cancelados e quantos possuem outro status.

print("Sem entrega e cancelados:", len(sem_entrega_cancelados))
print("Sem entrega com outro status:", len(sem_entrega_outros_status))

Sem entrega e cancelados: 109
Sem entrega com outro status: 34


In [ ]:
# Verificar se a hipótese é verdadeira ou falsa.

if len(sem_entrega_outros_status) == 0:
    print("Hipótese verdadeira.")
else:
    print("Hipótese falsa.")

#Falsa porque existem pedidos sem data de entrega que não estão cancelados.

Hipótese falsa.


In [36]:
# Mostrar alguns pedidos sem data de entrega que não estão cancelados.

display(
    sem_entrega_outros_status[
        [
            "pedido_id",
            "status_pedido",
            "data_entrega"
        ]
    ].head(3)
)

,pedido_id,status_pedido,data_entrega
49,1170,Concluído,NaN
117,990,Concluído,NaN
152,740,Concluído,NaN


In [37]:
# Importar datetime para converter os textos de data em datas reais.

from datetime import datetime

In [38]:
# Criar uma função para converter data_venda para o formato de data.

def converter_data_venda(valor):
    try:
        return datetime.strptime(
            valor,
            "%Y-%m-%d %H:%M:%S"
        )
    except (ValueError, TypeError):
        return None

In [40]:
# Aplicar a conversão em uma nova coluna e contar quantas datas de venda são inválidas.

df["data_venda_convertida"] = df["data_venda"].apply(
    converter_data_venda
)
datas_invalidas = df["data_venda_convertida"].isna().sum()

print("Datas de venda inválidas:", datas_invalidas)

Datas de venda inválidas: 4


In [41]:
# Criar colunas de ano, mês, dia e hora a partir da data convertida.

df["ano"] = df["data_venda_convertida"].apply(
    lambda x: x.year if pd.notna(x) else None
)

df["mes"] = df["data_venda_convertida"].apply(
    lambda x: x.month if pd.notna(x) else None
)

df["dia"] = df["data_venda_convertida"].apply(
    lambda x: x.day if pd.notna(x) else None
)

df["hora"] = df["data_venda_convertida"].apply(
    lambda x: x.hour if pd.notna(x) else None
)

In [43]:
# Criar uma coluna no formato ano-mês para agrupar os pedidos por período.

df["ano_mes"] = df["data_venda_convertida"].apply(
    lambda x: x.strftime("%Y-%m") if pd.notna(x) else None
)

# Conferir as novas colunas criadas a partir da data.

display(
    df[
        [
            "data_venda",
            "data_venda_convertida",
            "ano",
            "mes",
            "dia",
            "hora",
            "ano_mes"
        ]
    ].head()
)

,data_venda,data_venda_convertida,ano,mes,dia,hora,ano_mes
0,2025-11-21 18:02:35,2025-11-21 18:02:35,2025.0,11.0,21.0,18.0,2025-11
1,2024-10-29 18:41:26,2024-10-29 18:41:26,2024.0,10.0,29.0,18.0,2024-10
2,2025-09-05 21:16:09,2025-09-05 21:16:09,2025.0,9.0,5.0,21.0,2025-09
3,2024-11-08 11:29:44,2024-11-08 11:29:44,2024.0,11.0,8.0,11.0,2024-11
4,2025-08-23 11:22:05,2025-08-23 11:22:05,2025.0,8.0,23.0,11.0,2025-08


In [44]:
# Mostrar os registros cuja data de venda não pôde ser convertida.

registros_datas_invalidas = df[
    df["data_venda_convertida"].isna()
]

display(
    registros_datas_invalidas[
        [
            "pedido_id",
            "data_venda"
        ]
    ]
)

,pedido_id,data_venda
152,740,data_invalida
255,370,data_invalida
714,1480,data_invalida
884,1110,data_invalida


In [45]:
# Conferir quantos registros possuem data válida e inválida.

total_registros = len(df)

registros_datas_validas = df[
    "data_venda_convertida"
].notna().sum()

print("Total de registros:", total_registros)
print("Registros com data válida:", registros_datas_validas)
print("Registros com data inválida:", datas_invalidas)

Total de registros: 1500
Registros com data válida: 1496
Registros com data inválida: 4
